# **Data Analyst Job Postings: Dataset Cleaning**

### **About This Notebook**
This notebook cleans the raw `glassdoor-raw.csv` dataset scraped from Glassdoor and produces a fully structured, analysis-ready file `glassdoor-cleaned.csv`.

| Step | What Happens |
|:-:|---|
| 0 | Install and import all required libraries |
| 1 | Load the raw dataset and inspect its structure |
| 2 | Apply general cleaning to all columns |
| 3 | Fix two specific raw data issues found during inspection |
| 4 | Clean `industry` and `sector` for RQ1 |
| 5 | Parse `salary_estimate` into numeric columns for RQ2 |
| 6 | Derive `seniority_level` and extract skill flags for RQ3 |
| 7 | Final summary and save |



---
## **Step 0 - Imports**

We load all Python libraries needed throughout this notebook.

- `pandas` — loading, transforming, and saving tabular data
- `numpy` — numerical operations and null detection
- `re` — regular expressions for skill flag extraction
- `warnings` — suppress non-critical warnings for cleaner output

In [2]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

---
## **Step 1 — Load Raw Data**

We upload and load `glassdoor-raw.csv`, then inspect its shape, column data types, and where any null values appear.

> **Why inspect before cleaning?**
Understanding the raw structure before touching anything prevents accidental data loss and tells us exactly which columns need attention.

In [3]:
# @title
#Import file to Google Colab
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('glassdoor-raw.csv', index_col=0)
print(f'Shape: {df.shape}')
df.head(5)

Saving glassdoor-raw.csv to glassdoor-raw.csv
Shape: (2253, 15)


,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,Easy Apply
1352,Data Analyst,$30K-$53K (Glassdoor est.),"Job Description\nETL, SQL Queries, Data Modeli...",-1.0,1,"Dallas, TX",-1,-1,-1,-1,-1,-1,-1,-1,-1
119,PBM Data Analyst,$51K-$87K (Glassdoor est.),Responsibilities:\nProduce analyses and create...,3.2,1199SEIU Funds\n3.2,"New York, NY","New York, NY",1001 to 5000 employees,-1,Nonprofit Organization,Insurance Carriers,Insurance,$100 to $500 million (USD),-1,-1
324,Senior Data Analyst,$42K-$74K (Glassdoor est.),Responsibilities\nAssist the Associate Directo...,3.2,1199SEIU Funds\n3.2,"New York, NY","New York, NY",1001 to 5000 employees,-1,Nonprofit Organization,Insurance Carriers,Insurance,$100 to $500 million (USD),-1,-1
812,Data Governance Analyst,$42K-$76K (Glassdoor est.),SUMMARY\nThe Data Governance Analyst performs ...,5.0,1872 Consulting\n5.0,"Chicago, IL","Chicago, IL",1 to 50 employees,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,-1
1074,Data Analyst - Entry Level,$41K-$78K (Glassdoor est.),Looking to hire for a Data Analyst role with o...,-1.0,2000 east westmoreland st llc,"King of Prussia, PA",-1,-1,-1,-1,-1,-1,-1,-1,-1


## **Initial data types and missing values**

In [4]:
# @title
#Get the information of the dataset
print('\nDataset Info:')
print(df.info())

#Identify the row and column of null cells
row_indices, col_indices = np.where(pd.isnull(df))
print("\nCoordinates of null cells (row, column):")
for r, c in zip(row_indices, col_indices):
  print(f"  Row: {r}, Column: {c} (Column name: {df.columns[c]})")



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
Index: 2253 entries, 1352 to 1860
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Job Title          2253 non-null   object 
 1   Salary Estimate    2253 non-null   object 
 2   Job Description    2253 non-null   object 
 3   Rating             2253 non-null   float64
 4   Company Name       2252 non-null   object 
 5   Location           2253 non-null   object 
 6   Headquarters       2253 non-null   object 
 7   Size               2253 non-null   object 
 8   Founded            2253 non-null   int64  
 9   Type of ownership  2253 non-null   object 
 10  Industry           2253 non-null   object 
 11  Sector             2253 non-null   object 
 12  Revenue            2253 non-null   object 
 13  Competitors        2253 non-null   object 
 14  Easy Apply         2253 non-null   object 
dtypes: float64(1), int64(1), object(13)
memory usage: 281.6+ KB

---
## **Step 2 — General Cleaning**

Five actions applied to the entire dataset before any column-specific work.

**① Standardize column names:**
Column names like `Job Title` contain spaces and mixed casing. Lowercase everything and replace spaces with underscores so columns are consistent and easy to reference in code (e.g. `df['job_title']` instead of `df['Job Title']`).

**② Replace `-1` sentinel values with `NaN`:** Glassdoor uses `-1` as a scraping placeholder for missing data. We replace all occurrences with proper `NaN` so pandas treats them as missing values and they appear correctly in null checks.

**③ Remove duplicate rows:**
Duplicate rows skew counts and frequencies in any analysis. We check and remove any identical rows, then reset the index.

**④ Strip rating from company name:**
The scraper joined the company name and star rating in one field, (e.g. `'Vera Institute\n3.2'`). Split on `\n` and keep only the name, the rating already lives in its own column.

**⑤ Cast `founded` to nullable integer:**
Introducing `NaN` upcasts the column to `float64`, showing years as `1961.0`.
Cast to `Int64` to store whole numbers and `NaN` cleanly without decimals.

In [5]:
# @title
#Make all column names lowercase and replace " " with "_"
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Company Name — strip appended rating (e.g. 'Squarespace\n3.4' → 'Squarespace')
df['company_name'] = df['company_name'].str.split('\n').str[0].str.strip()

# Replace -1 sentinel values with NaN across all columns
df.replace(-1, np.nan, inplace=True)
df.replace('-1', np.nan, inplace=True)

# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Rows before: {before} | Rows after: {len(df)} | Duplicates removed: {before - len(df)}')

#Convert 'founded' to nullable integer
df['founded'] = df['founded'].astype('Int64')

#Check DataFrame
#print(df.info())
df

Rows before: 2253 | Rows after: 2253 | Duplicates removed: 0


,job_title,salary_estimate,job_description,rating,company_name,location,headquarters,size,founded,type_of_ownership,industry,sector,revenue,competitors,easy_apply
0,Data Analyst,$30K-$53K (Glassdoor est.),"Job Description\nETL, SQL Queries, Data Modeli...",NaN,1,"Dallas, TX",NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
1,PBM Data Analyst,$51K-$87K (Glassdoor est.),Responsibilities:\nProduce analyses and create...,3.2,1199SEIU Funds,"New York, NY","New York, NY",1001 to 5000 employees,<NA>,Nonprofit Organization,Insurance Carriers,Insurance,$100 to $500 million (USD),NaN,NaN
2,Senior Data Analyst,$42K-$74K (Glassdoor est.),Responsibilities\nAssist the Associate Directo...,3.2,1199SEIU Funds,"New York, NY","New York, NY",1001 to 5000 employees,<NA>,Nonprofit Organization,Insurance Carriers,Insurance,$100 to $500 million (USD),NaN,NaN
3,Data Governance Analyst,$42K-$76K (Glassdoor est.),SUMMARY\nThe Data Governance Analyst performs ...,5.0,1872 Consulting,"Chicago, IL","Chicago, IL",1 to 50 employees,<NA>,Company - Private,NaN,NaN,Unknown / Non-Applicable,NaN,NaN
4,Data Analyst - Entry Level,$41K-$78K (Glassdoor est.),Looking to hire for a Data Analyst role with o...,NaN,2000 east westmoreland st llc,"King of Prussia, PA",NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,Senior Business Technology Analyst - MDM / Dat...,$46K-$86K (Glassdoor est.),ZS is a professional services firm that works ...,3.8,ZS Associates,"San Francisco, CA","Evanston, IL",5001 to 10000 employees,1983,Company - Private,Consulting,Business Services,Unknown / Non-Applicable,NaN,NaN
2249,Senior Business Technology Analyst - MDM / Dat...,$47K-$74K (Glassdoor est.),ZS is a professional services firm that works ...,3.0,"ZS Associates, Inc.","San Francisco, CA","Evanston, IL",5001 to 10000 employees,<NA>,Company - Private,NaN,NaN,Unknown / Non-Applicable,NaN,NaN
2250,Data Analyst,$42K-$76K (Glassdoor est.),Zynga is a leading developer of the world’s mo...,4.1,Zynga,"Austin, TX","San Francisco, CA",1001 to 5000 employees,2007,Company - Public,Video Games,Media,$500 million to $1 billion (USD),NaN,NaN
2251,Data Analyst I,$65K-$81K (Glassdoor est.),Full-time Data Analyst to conduct research and...,NaN,"zz-Tarzana Treatment Centers, Inc.","Northridge, CA",NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN


---
## **Step 3 — RQ1: Clean `industry` and `sector`**

**Research Question 1** asks which industries and sectors show the highest hiring activity based on job posting frequency.

> **What needs cleaning:**
Both columns contain inconsistent casing — values like `it services`, `IT Services`, and `It Services` all refer to the same industry but would be counted as separate groups if left as-is. We apply `.str.title()` to normalize every value.

> **About the 353 missing rows:**
These were the `-1` sentinels replaced in Step 2. They have no industry or sector information and are excluded from RQ1 counts only — they remain in the dataset and still contribute to RQ2 and RQ3.

In [6]:
# @title
# Standardise casing
df['industry'] = df['industry'].str.strip().str.title()
df['sector']   = df['sector'].str.strip().str.title()

# Diagnostic — rows missing both Industry and Sector
rq1_missing = df[['industry', 'sector']].isnull().all(axis=1).sum()
print(f'Rows missing both Industry and Sector: {rq1_missing}')
print(f'Usable rows for RQ1: {len(df) - rq1_missing}')

Rows missing both Industry and Sector: 353
Usable rows for RQ1: 1900


In [7]:
# @title
# Preview
print('Industry value counts (top 10):')
print(df['industry'].value_counts().head(10))
print('\nSector value counts:')
print(df['sector'].value_counts())

Industry value counts (top 10):
industry
It Services                                325
Staffing & Outsourcing                     323
Health Care Services & Hospitals           151
Computer Hardware & Software               111
Consulting                                 111
Investment Banking & Asset Management       78
Enterprise Software & Network Solutions     69
Internet                                    65
Banks & Credit Unions                       51
Advertising & Marketing                     51
Name: count, dtype: int64

Sector value counts:
sector
Information Technology                570
Business Services                     524
Finance                               169
Health Care                           151
Education                              52
Insurance                              51
Accounting & Legal                     43
Media                                  42
Manufacturing                          40
Retail                                 38
Government    

---
## **Step 4 — RQ2: Parse `salary_estimate` into Numeric Columns**

**Research Question 2** asks which states offer the best combination of job demand and compensation.

> **The problem:** `salary_estimate` is stored as raw text like `$65K-$90K (Glassdoor est.)`. Math operations cannot be performed on text.

> **What we create:**
| New Column | Description | Example |
|---|---|---|
| `salary_min` | Lower bound, in dollars | 65,000 |
| `salary_max` | Upper bound, in dollars | 90,000 |
| `salary_avg` | Midpoint: `(min + max) / 2` | 77,500 |
| `salary_missing` | `True` if salary could not be parsed | False |

The regex extracts the number before `K` (e.g. `65` from `$65K`) and multiplies by 1,000 to get the full dollar value.

In [8]:
# @title
# Split Location into City and State
location_split    = df['location'].str.split(', ', n=1, expand=True)
df['city']        = location_split[0].str.strip()
df['state']       = location_split[1].str.strip()

print(f'Rows missing State: {df["state"].isnull().sum()}')
print('\nTop 10 states by posting count:')
print(df['state'].value_counts().head(10))

Rows missing State: 0

Top 10 states by posting count:
state
CA    626
TX    394
NY    345
IL    164
PA    114
AZ     97
NC     90
CO     88
NJ     86
WA     54
Name: count, dtype: int64


In [9]:
# @title
# Parse salary range into numeric columns
salary_clean         = df['salary_estimate'].str.replace(r'\(.*?\)', '', regex=True).str.strip()
df['salary_min'] = salary_clean.str.extract(r'\$(\d+)K')[0].astype(float) *1000
df['salary_max'] = salary_clean.str.extract(r'-\$(\d+)K')[0].astype(float) *1000
df['salary_avg'] = (df['salary_min'] + df['salary_max']) / 2
df['salary_missing'] = df['salary_avg'].isnull()

df['salary_min']
df['salary_max']

print('Salary Avg summary:')
print(df['salary_avg'].describe())

Salary Avg summary:
count      2252.000000
mean      72123.001776
std       23600.733829
min       33500.000000
25%       58000.000000
50%       69000.000000
75%       80500.000000
max      150000.000000
Name: salary_avg, dtype: float64


---
## **Step 5 — RQ3: Derive `seniority_level` and Extract Skill Flags**

**Research Question 3** asks how required technical skills vary across entry-level, mid-level, and senior Data Analyst roles.

**① `seniority_level`**
The raw dataset has no seniority column. We derive it by fusing three signals in order of reliability:

> **Seniority Classification**
| Level | S1: Title keywords & Roman numerals | S2: Years of experience | S3: Description language   |
|---|---|---|----|
| Senior      | `senior`, `sr.`, `lead`, `principal`,        | 6+ years                                     | "cross-functional", "mentor",                   |
|             | `staff`, `manager`, `Analyst III / IV`       |                                              | "architect", "10+ years"                        |
| Mid-Level   | —                                            | 3 – 5 years                                  | —                                               |
| Entry-Level | `entry`, `jr.`, `junior`, `associate`,       | ≤ 2 years                                    | "new graduate", "will train", "0–1 year"        |
|             | `intern`, `trainee`, `Analyst I`             |                                              |                                                 |
| Default     | —                                            | —                                            | Mid-Level (no signal found)                     |

**② Skill flags**
Skills are only mentioned inside the free-text `job_description` field. We use regular expressions to detect whether each of 9 skills appears in each description, creating a binary column per skill (`1` = mentioned, `0` = not mentioned).

> **Skills detected:** Python, SQL, Tableau, Excel, R, Power BI, Spark, Machine Learning, Statistics

In [13]:
# @title
import re

# SIGNAL 1 — Job Title keywords & Roman numerals
def title_seniority_score(title):
    title = str(title).lower()

    roman_map = {
        r'\bdata\s+\w*\s*analyst\s+i\b':   ('Entry-Level', 'high'),
        r'\bdata\s+\w*\s*analyst\s+ii\b':  ('Mid-Level',   'high'),
        r'\bdata\s+\w*\s*analyst\s+iii\b': ('Senior',      'high'),
        r'\bdata\s+\w*\s*analyst\s+iv\b':  ('Senior',      'high'),
        r'\blevel\s+1\b':                  ('Entry-Level', 'medium'),
        r'\blevel\s+2\b':                  ('Mid-Level',   'medium'),
        r'\blevel\s+3\b':                  ('Senior',      'medium'),
    }
    for pattern, result in roman_map.items():
        if re.search(pattern, title):
            return result

    senior_kw = ['senior', 'sr.', 'sr ', 'lead', 'principal', 'staff',
                 'manager', 'director', 'head of', 'vp ', 'vice president']
    entry_kw  = ['entry', 'jr.', 'jr ', 'junior', 'associate', 'intern',
                 'graduate', 'trainee', 'apprentice', 'early career']

    if any(kw in title for kw in senior_kw):
        return ('Senior', 'high')
    if any(kw in title for kw in entry_kw):
        return ('Entry-Level', 'high')
    return (None, None)


# SIGNAL 2 — Years of experience from Job Description
def extract_min_exp_years(text):
    text = str(text).lower()
    patterns = [
        r'minimum\s+of\s+(\d+)\s*(?:\+)?\s*years?',
        r'at\s+least\s+(\d+)\s*(?:\+)?\s*years?',
        r'(\d+)\+\s*years?\s*(?:of\s*)?(?:experience|exp)',
        r'(\d+)\s*-\s*\d+\s*years?\s*(?:of\s*)?(?:experience|exp)',
        r'(\d+)\s*years?\s*(?:of\s*)?(?:experience|exp)',
    ]
    for p in patterns:
        m = re.search(p, text)
        if m:
            val = int(m.group(1))
            if 0 < val <= 20:
                return val
    return None

def exp_years_to_level(years):
    if years is None:   return None
    if years <= 2:      return 'Entry-Level'
    if years <= 5:      return 'Mid-Level'
    return 'Senior'


# SIGNAL 3 — Description language (weak tiebreaker)
def desc_seniority_signal(text):
    text = str(text).lower()
    senior_phrases = ['mentor', 'lead a team', 'manage a team', 'strategic',
                      'cross-functional', 'architect', '8+ years', '10+ years']
    entry_phrases  = ['no experience required', 'new graduate', 'recent graduate',
                      'entry-level', '0-1 year', '0-2 year', 'will train']

    senior_hits = sum(1 for p in senior_phrases if p in text)
    entry_hits  = sum(1 for p in entry_phrases  if p in text)

    if senior_hits > entry_hits and senior_hits >= 2: return 'Senior'
    if entry_hits  > senior_hits and entry_hits >= 1: return 'Entry-Level'
    return None


# COMBINED CLASSIFIER — weighted signal fusion
def classify_seniority(row):
    title_level, title_conf = title_seniority_score(row['job_title'])
    exp_level  = exp_years_to_level(extract_min_exp_years(row['job_description']))
    desc_level = desc_seniority_signal(row['job_description'])

    if title_level and title_conf == 'high': return title_level  # most reliable
    if exp_level:                            return exp_level     # years of exp
    if title_level and title_conf == 'medium': return title_level
    if desc_level:                           return desc_level    # weak fallback
    return 'Mid-Level'                                           # default


df['seniority_level'] = df.apply(classify_seniority, axis=1)

print('Seniority Level distribution:')
print(df['seniority_level'].value_counts())

Seniority Level distribution:
seniority_level
Mid-Level      1245
Senior          673
Entry-Level     335
Name: count, dtype: int64


In [14]:
# @title
# Binary skill flags extracted from Job Description
skills = {
    'python':           r'\bpython\b',
    'sql':              r'\bsql\b',
    'tableau':          r'\btableau\b',
    'excel':            r'\bexcel\b',
    'r':                r'\b(?:r|r programming|r studio)\b',
    'power_bi':         r'\bpower\s?bi\b',
    'spark':            r'\bspark\b',
    'machine_learning': r'\bmachine\s?learning\b',
    'statistics':       r'\b(?:statistics|statistical)\b',
}

for skill, pattern in skills.items():
    df[f'skill_{skill}'] = df['job_description'].str.lower().str.contains(pattern, regex=True).astype(int)

skill_cols = [f'skill_{s}' for s in skills] # Corrected to match column creation
print('Skill mention counts across all postings:')
print(df[skill_cols].sum().sort_values(ascending=False))

Skill mention counts across all postings:
skill_sql                 1363
skill_excel                903
skill_statistics           836
skill_python               633
skill_tableau              617
skill_r                    441
skill_power_bi             246
skill_machine_learning     180
skill_spark                 71
dtype: int64


---
## **Step 6 — Final Dataset Summary and Save**

A complete overview of the cleaned dataset before saving. We confirm the shape, review remaining nulls, and preview the result.

In [15]:
# @title
print(f'Final shape: {df.shape}')
print(f'\nColumns ({len(df.columns)}):')
for col in df.columns:
    print(f'  - {col}')

Final shape: (2253, 31)

Columns (31):
  - job_title
  - salary_estimate
  - job_description
  - rating
  - company_name
  - location
  - headquarters
  - size
  - founded
  - type_of_ownership
  - industry
  - sector
  - revenue
  - competitors
  - easy_apply
  - city
  - state
  - salary_min
  - salary_max
  - salary_avg
  - salary_missing
  - seniority_level
  - skill_python
  - skill_sql
  - skill_tableau
  - skill_excel
  - skill_r
  - skill_power_bi
  - skill_spark
  - skill_machine_learning
  - skill_statistics


In [16]:
# @title
print('Missing values (columns with any NaN):')
missing = df.isnull().sum()
print(missing[missing > 0])

Missing values (columns with any NaN):
salary_estimate         1
rating                272
company_name            1
headquarters          172
size                  163
founded               660
type_of_ownership     163
industry              353
sector                353
revenue               163
competitors          1732
easy_apply           2173
salary_min              1
salary_max              1
salary_avg              1
dtype: int64


In [17]:
df.head(5)

,job_title,salary_estimate,job_description,rating,company_name,location,headquarters,size,founded,type_of_ownership,...,seniority_level,skill_python,skill_sql,skill_tableau,skill_excel,skill_r,skill_power_bi,skill_spark,skill_machine_learning,skill_statistics
0,Data Analyst,$30K-$53K (Glassdoor est.),"Job Description\nETL, SQL Queries, Data Modeli...",NaN,1,"Dallas, TX",NaN,NaN,<NA>,NaN,...,Mid-Level,0,1,0,0,0,0,0,0,0
1,PBM Data Analyst,$51K-$87K (Glassdoor est.),Responsibilities:\nProduce analyses and create...,3.2,1199SEIU Funds,"New York, NY","New York, NY",1001 to 5000 employees,<NA>,Nonprofit Organization,...,Mid-Level,0,1,0,1,0,0,0,0,1
2,Senior Data Analyst,$42K-$74K (Glassdoor est.),Responsibilities\nAssist the Associate Directo...,3.2,1199SEIU Funds,"New York, NY","New York, NY",1001 to 5000 employees,<NA>,Nonprofit Organization,...,Senior,0,1,0,1,0,0,0,0,1
3,Data Governance Analyst,$42K-$76K (Glassdoor est.),SUMMARY\nThe Data Governance Analyst performs ...,5.0,1872 Consulting,"Chicago, IL","Chicago, IL",1 to 50 employees,<NA>,Company - Private,...,Entry-Level,0,0,0,0,0,0,0,0,1
4,Data Analyst - Entry Level,$41K-$78K (Glassdoor est.),Looking to hire for a Data Analyst role with o...,NaN,2000 east westmoreland st llc,"King of Prussia, PA",NaN,NaN,<NA>,NaN,...,Entry-Level,0,0,0,0,0,0,0,0,1


---
## **Step 7 — Save Cleaned Dataset**



In [18]:
df.to_csv('glassdoor-cleaned.csv', index=False)
files.download('glassdoor-cleaned.csv')
print('Saved: glassdoor-cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved: glassdoor-cleaned.csv


### **Cleaning Summary Table**

| Step | Column(s) | Issue | Action | Result |
|:-:|---|---|---|---|
| 2 | All | Column names with spaces and mixed case | Lowercase + underscores | Consistent naming |
| 2 | All | `-1` used as null placeholder | Replaced with `NaN` | Proper missing values |
| 2 | All | Duplicate rows | `drop_duplicates()` | 0 removed |
| 2 | `company_name` | Glassdoor rating embedded via `\n` | Strip after `\n` | Clean company names |
| 2 | `location` | City and state bundled in one column | Split into `city` + `state` | Two usable columns |
| 2 | `founded` | Became `float64` after NaN replacement | Cast to `Int64` | No more decimal years |
| 3 | `industry`, `sector` | Mixed casing | `.str.title()` | Consistent labels |
| 4 | `salary_estimate` | Text range, not numeric | Parsed to `salary_min`, `salary_max`, `salary_avg` | 3 numeric salary columns |
| 5 | `job_title` | No seniority column in raw data | Keyword classification | `seniority_level` column |
| 5 | `job_description` | 9 skills unstructured in free text | Regex extraction | 9 `skill_*` binary columns |

**Raw:** 2,253 rows x 15 columns  
**Cleaned:** 2,253 rows x 30 columns  